In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

os.chdir(r"C:\Users\PREM\OneDrive\Desktop\Bluestock_MF_Capstone")

RAW  = Path("data/raw")
PROC = Path("data/processed")
PROC.mkdir(exist_ok=True)

print("Setup complete!")

Setup complete!


In [2]:
# ── Load raw files ─────────────────────────────────────────────
nav_history  = pd.read_csv(RAW / "02_nav_history.csv")
transactions = pd.read_csv(RAW / "08_investor_transactions.csv")
perf         = pd.read_csv(RAW / "07_scheme_performance.csv")

print("Files loaded!")

Files loaded!


In [3]:
nav_history["date"] = pd.to_datetime(nav_history["date"])
nav_history = nav_history.sort_values(["amfi_code", "date"])

before = len(nav_history)
nav_history = nav_history.drop_duplicates(subset=["amfi_code", "date"])
print(f"Duplicates removed  : {before - len(nav_history)}")

nav_history = nav_history[nav_history["nav"] > 0]

nav_history["daily_return_pct"] = (
    nav_history
    .groupby("amfi_code")["nav"]
    .pct_change()
    .mul(100)
    .round(4)
)

print(f"Final shape         : {nav_history.shape}")
print(f"Date range          : {nav_history['date'].min().date()} → {nav_history['date'].max().date()}")
nav_history.head()

Duplicates removed  : 0
Final shape         : (46000, 4)
Date range          : 2022-01-03 → 2026-05-29


,amfi_code,date,nav,daily_return_pct
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-1.0306
5752,100016,2022-01-05,521.7239,1.2865
5753,100016,2022-01-06,515.7880,-1.1377
5754,100016,2022-01-07,515.1639,-0.1210


In [4]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["transaction_type"] = transactions["transaction_type"].str.strip().str.title()

before = len(transactions)
transactions = transactions[transactions["amount_inr"] > 0]
print(f"Invalid rows removed : {before - len(transactions)}")
print(f"Transaction types    : {transactions['transaction_type'].unique()}")
print(f"KYC status values    : {transactions['kyc_status'].unique()}")
print(f"Final shape          : {transactions.shape}")
transactions.head()

Invalid rows removed : 0
Transaction types    : ['Sip' 'Redemption' 'Lumpsum']
KYC status values    : ['Verified' 'Pending']
Final shape          : (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,Sip,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,Sip,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,Sip,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [5]:
number_cols = [
    "return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
    "sharpe_ratio", "alpha", "beta",
    "max_drawdown_pct", "expense_ratio_pct"
]
for col in number_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")

print(f"Missing values:\n{perf.isnull().sum()[perf.isnull().sum() > 0]}")
print(f"Final shape : {perf.shape}")
perf.head()

Missing values:
Series([], dtype: int64)
Final shape : (40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [7]:
nav_history.to_csv(PROC  / "clean_nav.csv",         index=False)
transactions.to_csv(PROC / "clean_transactions.csv", index=False)
perf.to_csv(PROC         / "clean_performance.csv",  index=False)

print("All cleaned files saved!")
print(f"  clean_nav.csv          → {nav_history.shape}")
print(f"  clean_transactions.csv → {transactions.shape}")
print(f"  clean_performance.csv  → {perf.shape}")

All cleaned files saved!
  clean_nav.csv          → (46000, 4)
  clean_transactions.csv → (32778, 13)
  clean_performance.csv  → (40, 19)
